In [1]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [2]:
def wind_stress_curl(taux,tauy):
    _, tauxdy = mp.geospatial_gradient(taux)
    tauydx, _ = mp.geospatial_gradient(tauy)
    return(tauydx - tauxdy)

In [ ]:
## +++ LOAD DATA +++ ##

## Surface ocean eastward currents
## E2.1 Coupled Simulations
uocn=xr.open_dataset(f'{dpath0}/model_output/e2pt1/ocn.16m.currents.cpld.e2.1.nc').u.squeeze(drop=True) * 0.01
uocn=longitude_flip(uocn)
lato=uocn.lato
lono=uocn.lono

## Oscar obs
uobs=xr.open_dataset(f'{dpath1}/satellite/obs.oscar.u.nc').U.squeeze(drop=True)
uobs=uobs.where(uobs>-1e32)
# match model resolution and dim names
uobs=uobs.rename({'LONGITUDE':'lono', 'LATITUDE':'lato'})
uobs_lr=uobs.interp_like(uocn,method='linear')

## FLOR
uflorctrl=xr.open_dataset(f'{dpath0}/model_output/flor/flor.ctrl.u.nc').u.squeeze(drop=True)
uflorctrl=uflorctrl.groupby("time.month").mean()

uflorhi=xr.open_dataset(f'{dpath0}/model_output/flor/flor.hitopo.u.nc').u.squeeze(drop=True)
uflorhi=uflorhi.groupby("time.month").mean()


## Surface Wind Stresses
# MERRA-2
filen='/Users/dervlamk/OneDrive/research/obs_data/reanalysis/merra2/obs.merra2.taux-tauy.nc'
taux_obs=xr.open_dataset(f'{filen}').TAUXWTR.mean(dim='time')
tauy_obs=xr.open_dataset(f'{filen}').TAUYWTR.mean(dim='time')

# FLOR
dpath='model_output/flor'
taux_fc=xr.open_dataset(f'{dpath0}/{dpath}/flor.ctrl.taux.nc').tau_x.mean(dim='time')
tauy_fc=xr.open_dataset(f'{dpath0}/{dpath}/flor.ctrl.tauy.nc').tau_y.mean(dim='time')
taux_fh=xr.open_dataset(f'{dpath0}/{dpath}/flor.hitopo.taux.nc').tau_x.mean(dim='time')
tauy_fh=xr.open_dataset(f'{dpath0}/{dpath}/flor.hitopo.tauy.nc').tau_y.mean(dim='time')

# E2.1
dpath='/model_output/e2pt1'
e2pt1_taux={ 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }
e2pt1_tauy={ 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }
for key in e2pt1_taux.keys():
    e2pt1_taux[key] = xr.open_dataset(f'{dpath0}/{dpath}/aogcm.{key}.taux.nc').taux.mean(dim='time')
    e2pt1_tauy[key] = xr.open_dataset(f'{dpath0}/{dpath}/aogcm.{key}.tauy.nc').tauy.mean(dim='time')


In [ ]:
## +++ CALCULATE WIND STRESS CURL +++ ##

# MERRA-2 REANALYSIS
lat=taux_obs.lat
lon=taux_obs.lon
curl_obs=wind_stress_curl(taux_obs,tauy_obs)
curl_obs=xr.DataArray(data=curl_obs,
                      dims=['lat','lon'],
                      coords=dict(lat=lat,lon=lon,),
                      attrs=dict(description='Wind Stress Curl from tau_x and tau_y',
                                 units='N m^-3',
                                 source='MERRA-2',),)

# CM2.5-FLOR SIMULATIONS
lat=taux_fc.yu_ocean
lon=taux_fc.xu_ocean
curl_fc=wind_stress_curl(taux_fc,tauy_fc)
curl_fc=xr.DataArray(data=curl_fc,
                      dims=['yu_ocean','xu_ocean'],
                      coords=dict(yu_ocean=lat,xu_ocean=lon,),
                      attrs=dict(description='Wind Stress Curl from tau_x and tau_y',
                                 units='N m^-3',
                                 source='FLOR CTRL',),)
curl_fh=wind_stress_curl(taux_fh,tauy_fh)
curl_fh=xr.DataArray(data=curl_fh,
                      dims=['yu_ocean','xu_ocean'],
                      coords=dict(yu_ocean=lat,xu_ocean=lon,),
                      attrs=dict(description='Wind Stress Curl from tau_x and tau_y',
                                 units='N m^-3',
                                 source='FLOR HiTOPO',),)

# GISS E2.1 SIMULATIONS
lat=e2pt1_taux['lat'].lat
lon=e2pt1_taux['lon'].lon
e2pt1_curl={ 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }
for key in e2pt1_curl.keys():
    curl_g = wind_stress_curl(e2pt1_taux[key], e2pt1_tauy[key])/1000 # convert g to kg
    e2pt1_curl[key]=xr.DataArray(data=curl_g,
                                 dims=['lat','lon'],
                                 coords=dict(lat=lat,lon=lon,),
                                 attrs=dict(description='Wind Stress Curl from taux and tauy',
                                            source=f'E2.1 {key}',),)

In [ ]:
## meridional derivative of zonal integral between 170°W--110°W
# set bounds
xmin=-170; xmax=-110
ymin=-5;   ymax=20

deriv_g=curl_e.sel(lon=slice(xmin,xmax), lat=slice(ymin,ymax)).integrate('lon').differentiate('lat')
deriv_obs=curl_obs.sel(lon=slice(xmin,xmax), lat=slice(ymin,ymax)).integrate('lon').differentiate('lat')
deriv_fc=curl_fc.sel(xu_ocean=slice(xmin,xmax), yu_ocean=slice(ymin,ymax)).integrate('xu_ocean').differentiate('yu_ocean')
deriv_fh=curl_fh.sel(xu_ocean=slice(xmin,xmax), yu_ocean=slice(ymin,ymax)).integrate('xu_ocean').differentiate('yu_ocean')

In [ ]:
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 8, 'labelcolor': 'k'}   
text_kw={'color':'k', 'weight':'normal', 'size':8, 'ha':'center'}
text_kw2={'color':'k', 'weight':'bold', 'size':10, 'ha':'left'}
legend_kw={'fontsize':7, 'ncols':1,  'loc':'upper left', 'bbox_to_anchor':(0, 1.0), 'labelcolor':'linecolor', 'frameon':False}

line_kw={'lw':1} 

fig=plt.figure(figsize=(6,6))

ax=plt.subplot2grid((1, 2), (0, 0), colspan=1, rowspan=1)
ax.text(-0.65, 15.1, 'U-CURRENT', **text_kw2)
ax.axvline(x=0, c='k', ls='-', lw=0.5)
ax.plot(uobs_zm.mean(dim='lono'), uobs_zm.lato, c='k', ls='-', label='OBS', **line_kw)
ax.plot(uflorctrl_zm.mean(dim='xu_ocean'), uflorctrl_zm.yu_ocean, c='r', ls='-', label='CM2.5-FLOR CTRL', **line_kw)
ax.plot(uflorhi_zm.mean(dim='xu_ocean'), uflorhi_zm.yu_ocean, c='r', ls='--', label='CM2.5-FLOR MAX', **line_kw)
ax.plot(uocn_zm[0].mean(dim='lono'), uocn_zm.lato, c='blue', ls='-', label='E2.1 CTRL', **line_kw)
ax.plot(uocn_zm[1].mean(dim='lono'), uocn_zm.lato, c='blue', ls=':', label='E2.1 CA-ONLY', **line_kw)
ax.plot(uocn_zm[2].mean(dim='lono'), uocn_zm.lato, c='blue', ls='--', label='E2.1 CA+SA', **line_kw)
# set axis info
ax.set(xlim=[-0.65, 0.2], ylim=[0,15])
ax.set_xlabel('[cm/s]', **text_kw)
ax.set_ylabel('LATITUDE [°N]', **text_kw)
ax.tick_params(**tkw)
ax.legend(**legend_kw) 


ax=plt.subplot2grid((1, 2), (0, 1), colspan=1, rowspan=1)
ax.text(-12, 15.1, '$\delta$ WIND STRESS CURL', **text_kw2)
ax.axvline(x=0, c='k', ls='-', lw=0.5)
ax.plot(deriv_obs*1e7, deriv_obs.lat, c='k', ls='-', label='OBS', **line_kw)
ax.plot(deriv_fc*1e7, deriv_fc.yu_ocean, c='red', ls='-', label='FLOR CTRL', **line_kw)
ax.plot(deriv_fh*1e7, deriv_fh.yu_ocean, c='red', ls='--', label='FLOR MAX', **line_kw)

ax.plot(deriv_g[0]*1e7, deriv_g.lat, c='blue', ls='-', label='E2.1 CTRL', **line_kw)
ax.plot(deriv_g[1]*1e7, deriv_g.lat, c='blue', ls=':', label='E2.1 CA-ONLY', **line_kw)
ax.plot(deriv_g[2]*1e7, deriv_g.lat, c='blue', ls='--', label='E2.1 CA+SA', **line_kw)
# set axis info
ax.set(xlim=[-12, 15], ylim=[0,15])
ax.set_xlabel('[N m$^{-3}$]', **text_kw)
ax.tick_params(**tkw)
